In [ ]:
# 安装必要的库（如果尚未安装）
# 如果库已安装，可以跳过此cell
import subprocess
import sys

def install_package(package):
    """安装Python包"""
    try:
        __import__(package)
        print(f"✅ {package} 已安装")
    except ImportError:
        print(f"📦 正在安装 {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} 安装完成")

# 安装所需的库
packages = [
    "pandas",
    "matplotlib",
    "numpy",
    "scipy",
    "jupyter"
]

print("=" * 60)
print("检查并安装必要的Python库")
print("=" * 60)

for package in packages:
    install_package(package)

print("\n" + "=" * 60)
print("✅ 所有库检查完成！")
print("=" * 60)

# Pareto前沿学术论文对比分析

本Notebook专门用于学术论文的第一代和最后一代Pareto前沿对比分析。

## 功能
- **第一代和最后一代的Pareto前沿对比可视化**（2D和3D）
- **学术论文级别的定量评估指标**：超体积(HV)、世代距离(GD)、反向世代距离(IGD)、分布性(Spread)
- **统计显著性检验**：t检验
- **批量生成统计表格**：适合直接用于论文
- **支持3个或4个目标**：自动检测目标数量，支持新旧两种格式

## 使用方法
1. 确保已运行实验并生成了第一代和最后一代的Pareto前沿数据
2. 修改相应的参数（实验目录、调度器名称、trial编号等）
3. 运行相应的代码单元格

## 目标函数说明

### 新格式（推荐）
- **Makespan**: 最大完成时间（越小越好）
- **CostEfficiency**: 成本效率比（总成本/总工作量）（越小越好）
- **LoadBalanceIndex**: 负载均衡指数，归一化的负载不均衡度（越小越好）
- **ResourceWaste**: 资源浪费率（1 - 加权平均利用率）（越小越好）

### 旧格式（向后兼容）
- **Makespan**: 最大完成时间（越小越好）
- **Cost**: 执行成本（越小越好）
- **LoadBalance**: 负载均衡度，使用变异系数（CV）（越小越好）
- **ResourceUtilization**: 资源利用率，转换为最小化目标（1 - utilization）（越小越好）

**注意**：Notebook会自动检测CSV文件中的目标列名称，支持新旧两种格式的向后兼容。

In [ ]:
# 导入必要的库
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 库导入成功！")

In [ ]:
# 辅助函数：加载Pareto前沿数据
def load_pareto_front(file_path):
    """
    加载Pareto前沿数据，并自动去重（去除目标值完全相同的重复解）
    
    参数:
        file_path: CSV文件路径
    
    返回:
        DataFrame: 去重后的Pareto前沿数据
    """
    try:
        df = pd.read_csv(file_path)
        if df.empty:
            return df
        
        # 检测可用的目标列
        obj_cols = get_objective_columns(df)
        if len(obj_cols) == 0:
            return df
        
        # 记录去重前的数量
        original_count = len(df)
        
        # 基于目标列去重（保留第一个出现的解）
        df_unique = df.drop_duplicates(subset=obj_cols, keep='first')
        
        # 如果去除了重复解，输出提示
        duplicate_count = original_count - len(df_unique)
        if duplicate_count > 0:
            print(f"  ⚠️ 去除了 {duplicate_count} 个重复的Pareto解（目标值完全相同）")
            print(f"  ✅ 保留 {len(df_unique)} 个不同的解（原始 {original_count} 个）")
        
        return df_unique
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

# 辅助函数：检测可用的目标列
def get_objective_columns(df):
    """
    检测DataFrame中可用的目标列
    支持新旧两种格式：
    - 新格式：Makespan, CostEfficiency, LoadBalanceIndex, ResourceWaste
    - 旧格式：Makespan, Cost, LoadBalance, ResourceUtilization
    
    返回:
        list: 可用的目标列名列表（按顺序：Makespan, Cost/CostEfficiency, LoadBalance/LoadBalanceIndex, ResourceUtilization/ResourceWaste）
    """
    # 新格式（推荐）
    new_format = {
        'Makespan': 'Makespan',
        'Cost': 'CostEfficiency',
        'LoadBalance': 'LoadBalanceIndex',
        'ResourceUtilization': 'ResourceWaste'
    }
    
    # 旧格式（向后兼容）
    old_format = {
        'Makespan': 'Makespan',
        'Cost': 'Cost',
        'LoadBalance': 'LoadBalance',
        'ResourceUtilization': 'ResourceUtilization'
    }
    
    # 检测使用哪种格式（优先检测新格式）
    available_cols = []
    
    # 首先检查新格式
    if 'CostEfficiency' in df.columns:
        # 使用新格式
        if 'Makespan' in df.columns:
            available_cols.append('Makespan')
        if 'CostEfficiency' in df.columns:
            available_cols.append('CostEfficiency')
        if 'LoadBalanceIndex' in df.columns:
            available_cols.append('LoadBalanceIndex')
        if 'ResourceWaste' in df.columns:
            available_cols.append('ResourceWaste')
    elif 'Cost' in df.columns:
        # 使用旧格式（向后兼容）
        if 'Makespan' in df.columns:
            available_cols.append('Makespan')
        if 'Cost' in df.columns:
            available_cols.append('Cost')
        if 'LoadBalance' in df.columns:
            available_cols.append('LoadBalance')
        if 'ResourceUtilization' in df.columns:
            available_cols.append('ResourceUtilization')
    else:
        # 如果都没有，尝试检测Makespan
        if 'Makespan' in df.columns:
            available_cols.append('Makespan')
    
    return available_cols

## 1. 学术论文评估指标函数

In [ ]:
def calculate_hypervolume(pareto_front, reference_point=None):
    """
    计算超体积指标 (Hypervolume Indicator)
    
    参数:
        pareto_front: DataFrame，包含目标列（支持新旧两种格式）
          新格式：Makespan, CostEfficiency, LoadBalanceIndex, ResourceWaste
          旧格式：Makespan, Cost, LoadBalance, ResourceUtilization
        reference_point: 参考点，如果为None则自动计算
    
    返回:
        hypervolume值（越大越好）
    """
    if pareto_front.empty:
        return 0.0
    
    # 自动检测可用的目标列
    obj_cols = get_objective_columns(pareto_front)
    if len(obj_cols) == 0:
        return 0.0
    
    objectives = pareto_front[obj_cols].values
    
    # 如果没有提供参考点，使用最差值作为参考点
    if reference_point is None:
        reference_point = objectives.max(axis=0) * 1.1  # 增加10%的余量
    
    # 归一化到[0,1]
    min_vals = objectives.min(axis=0)
    max_vals = objectives.max(axis=0)
    ranges = max_vals - min_vals
    ranges[ranges == 0] = 1  # 避免除零
    
    normalized = (objectives - min_vals) / ranges
    ref_normalized = (reference_point - min_vals) / ranges
    
    # 计算每个解到参考点的体积
    volumes = []
    for point in normalized:
        # 计算该点与参考点形成的超立方体体积
        volume = np.prod(np.maximum(0, ref_normalized - point))
        volumes.append(volume)
    
    if len(volumes) == 0:
        return 0.0
    
    return np.sum(volumes) / len(volumes) * len(pareto_front)

def calculate_generational_distance(pareto_front, reference_front=None):
    """
    计算世代距离 (Generational Distance, GD)
    
    参数:
        pareto_front: 当前Pareto前沿
        reference_front: 参考Pareto前沿（如果为None，使用第一代作为参考）
    
    返回:
        GD值（越小越好，0表示完全收敛）
    """
    if pareto_front.empty:
        return float('inf')
    
    # 自动检测可用的目标列
    obj_cols = get_objective_columns(pareto_front)
    if len(obj_cols) == 0:
        return float('inf')
    
    current = pareto_front[obj_cols].values
    
    if reference_front is None or reference_front.empty:
        # 如果没有参考前沿，使用理想点（各目标的最小值）
        ideal_point = current.min(axis=0)
        distances = np.sqrt(np.sum((current - ideal_point)**2, axis=1))
        return np.mean(distances)
    
    # 确保两个前沿使用相同的目标列
    ref_obj_cols = get_objective_columns(reference_front)
    common_cols = [col for col in obj_cols if col in ref_obj_cols]
    if len(common_cols) == 0:
        return float('inf')
    
    current = pareto_front[common_cols].values
    reference = reference_front[common_cols].values
    
    # 如果参考前沿只有一个解，且当前前沿也只有一个解，直接计算距离
    if len(reference) == 1 and len(current) == 1:
        # 归一化
        all_points = np.vstack([current, reference])
        min_vals = all_points.min(axis=0)
        max_vals = all_points.max(axis=0)
        ranges = max_vals - min_vals
        ranges[ranges == 0] = 1
        
        current_norm = (current - min_vals) / ranges
        reference_norm = (reference - min_vals) / ranges
        return np.sqrt(np.sum((current_norm - reference_norm)**2))
    
    # 归一化：使用所有点的范围
    all_points = np.vstack([current, reference])
    min_vals = all_points.min(axis=0)
    max_vals = all_points.max(axis=0)
    ranges = max_vals - min_vals
    
    # 如果某个目标的范围为0，说明所有解在该目标上相同，跳过该维度
    valid_dims = ranges > 1e-10
    if not np.any(valid_dims):
        return 0.0  # 所有目标都相同，距离为0
    
    # 只对有效维度进行归一化
    current_valid = current[:, valid_dims]
    reference_valid = reference[:, valid_dims]
    min_vals_valid = min_vals[valid_dims]
    max_vals_valid = max_vals[valid_dims]
    ranges_valid = ranges[valid_dims]
    
    current_norm = (current_valid - min_vals_valid) / ranges_valid
    reference_norm = (reference_valid - min_vals_valid) / ranges_valid
    
    # 计算每个当前解到最近参考解的距离
    distances = []
    for point in current_norm:
        dists = np.sqrt(np.sum((reference_norm - point)**2, axis=1))
        distances.append(dists.min())
    
    return np.mean(distances) if distances else 0.0

def calculate_igd(pareto_front, reference_front):
    """
    计算反向世代距离 (Inverted Generational Distance, IGD)
    
    参数:
        pareto_front: 当前Pareto前沿
        reference_front: 参考Pareto前沿
    
    返回:
        IGD值（越小越好）
    """
    if pareto_front.empty or reference_front.empty:
        return float('inf')
    
    # 自动检测可用的目标列
    obj_cols = get_objective_columns(pareto_front)
    ref_obj_cols = get_objective_columns(reference_front)
    common_cols = [col for col in obj_cols if col in ref_obj_cols]
    if len(common_cols) == 0:
        return float('inf')
    
    current = pareto_front[common_cols].values
    reference = reference_front[common_cols].values
    
    # 如果参考前沿只有一个解，且当前前沿也只有一个解，直接计算距离
    if len(reference) == 1 and len(current) == 1:
        # 归一化
        all_points = np.vstack([current, reference])
        min_vals = all_points.min(axis=0)
        max_vals = all_points.max(axis=0)
        ranges = max_vals - min_vals
        ranges[ranges == 0] = 1
        
        current_norm = (current - min_vals) / ranges
        reference_norm = (reference - min_vals) / ranges
        return np.sqrt(np.sum((current_norm - reference_norm)**2))
    
    # 归一化：使用所有点的范围
    all_points = np.vstack([current, reference])
    min_vals = all_points.min(axis=0)
    max_vals = all_points.max(axis=0)
    ranges = max_vals - min_vals
    
    # 如果某个目标的范围为0，说明所有解在该目标上相同，跳过该维度
    valid_dims = ranges > 1e-10
    if not np.any(valid_dims):
        return 0.0  # 所有目标都相同，距离为0
    
    # 只对有效维度进行归一化
    current_valid = current[:, valid_dims]
    reference_valid = reference[:, valid_dims]
    min_vals_valid = min_vals[valid_dims]
    max_vals_valid = max_vals[valid_dims]
    ranges_valid = ranges[valid_dims]
    
    current_norm = (current_valid - min_vals_valid) / ranges_valid
    reference_norm = (reference_valid - min_vals_valid) / ranges_valid
    
    # 计算每个参考解到最近当前解的距离
    distances = []
    for point in reference_norm:
        dists = np.sqrt(np.sum((current_norm - point)**2, axis=1))
        distances.append(dists.min())
    
    return np.mean(distances) if distances else 0.0

def calculate_spread(pareto_front):
    """
    计算分布性指标 (Spread)
    
    参数:
        pareto_front: Pareto前沿
    
    返回:
        Spread值（越小越好，0表示完全均匀分布）
    """
    if pareto_front.empty or len(pareto_front) < 2:
        return 0.0
    
    # 自动检测可用的目标列
    obj_cols = get_objective_columns(pareto_front)
    if len(obj_cols) == 0:
        return 0.0
    
    objectives = pareto_front[obj_cols].values
    
    # 归一化
    min_vals = objectives.min(axis=0)
    max_vals = objectives.max(axis=0)
    ranges = max_vals - min_vals
    ranges[ranges == 0] = 1
    
    normalized = (objectives - min_vals) / ranges
    
    # 计算相邻解之间的距离
    n = len(normalized)
    distances = []
    
    for i in range(n):
        min_dist = float('inf')
        for j in range(n):
            if i != j:
                dist = np.sqrt(np.sum((normalized[i] - normalized[j])**2))
                min_dist = min(min_dist, dist)
        distances.append(min_dist)
    
    # 计算平均距离
    mean_dist = np.mean(distances)
    
    # 计算标准差作为Spread指标
    if mean_dist == 0:
        return 0.0
    
    spread = np.std(distances) / mean_dist if mean_dist > 0 else 0.0
    
    return spread

print("✅ 评估指标函数已定义")

## 2. 第一代和最后一代对比可视化（2D）

In [ ]:
# 配置参数
experiment_dir = "../results/run_21"  # 修改为你的实验目录
scheduler_name = "MOGWO"  # 修改为调度器名称
trial = 1  # 修改为trial编号

# 加载第一代和最后一代的Pareto前沿数据
pareto_dir = os.path.join(experiment_dir, 'pareto_fronts')
file_first = os.path.join(pareto_dir, f"{scheduler_name}_trial_{trial}_first.csv")
file_final = os.path.join(pareto_dir, f"{scheduler_name}_trial_{trial}_final.csv")

# 检查文件是否存在
if not os.path.exists(file_first):
    print(f"⚠️ 第一代数据文件不存在: {file_first}")
    print("请确保已经运行过实验并生成了第一代Pareto前沿数据")
elif not os.path.exists(file_final):
    print(f"⚠️ 最后一代数据文件不存在: {file_final}")
    print("请确保已经运行过实验并生成了最后一代Pareto前沿数据")
else:
    df_first = load_pareto_front(file_first)
    df_final = load_pareto_front(file_final)
    
    if df_first is None or df_first.empty:
        print(f"⚠️ 第一代数据为空: {file_first}")
    elif df_final is None or df_final.empty:
        print(f"⚠️ 最后一代数据为空: {file_final}")
    else:
        print(f"✅ 第一代Pareto解数量: {len(df_first)}")
        print(f"✅ 最后一代Pareto解数量: {len(df_final)}")
        
        # 检测可用的目标列
        obj_cols = get_objective_columns(df_first)
        
        # 获取列名映射（用于显示标签）
        col_labels = {
            'Makespan': 'Makespan (完成时间)',
            'Cost': 'Cost (执行成本)',
            'CostEfficiency': 'CostEfficiency (成本效率比)',
            'LoadBalance': 'LoadBalance (负载均衡度)',
            'LoadBalanceIndex': 'LoadBalanceIndex (负载均衡指数)',
            'ResourceUtilization': 'ResourceUtilization (资源利用率)',
            'ResourceWaste': 'ResourceWaste (资源浪费率)'
        }
        
        # 根据目标数量创建子图
        if len(obj_cols) == 4:
            # 4个目标：创建2x3的网格（6个子图，所有两两组合）
            fig, axes = plt.subplots(2, 3, figsize=(18, 10))
            axes = axes.flatten()
            
            # 生成所有两两组合
            pairs = []
            for i in range(len(obj_cols)):
                for j in range(i + 1, len(obj_cols)):
                    pairs.append((obj_cols[i], obj_cols[j]))
            
            # 绘制每个组合
            for idx, (col1, col2) in enumerate(pairs[:6]):  # 最多6个组合
                axes[idx].scatter(df_first[col1], df_first[col2], 
                               alpha=0.5, s=40, color='red', label='第一代', marker='o')
                axes[idx].scatter(df_final[col1], df_final[col2], 
                               alpha=0.6, s=50, color='blue', label='最后一代', marker='^')
                axes[idx].set_xlabel(col_labels.get(col1, col1), fontsize=12)
                axes[idx].set_ylabel(col_labels.get(col2, col2), fontsize=12)
                axes[idx].set_title(f'{col1} vs {col2}', fontsize=14)
                axes[idx].legend()
                axes[idx].grid(True, alpha=0.3)
        else:
            # 3个目标：创建1x3的网格（3个子图，向后兼容）
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            
            # Makespan vs Cost/CostEfficiency
            col1, col2 = obj_cols[0], obj_cols[1]
            axes[0].scatter(df_first[col1], df_first[col2], 
                           alpha=0.5, s=40, color='red', label='第一代', marker='o')
            axes[0].scatter(df_final[col1], df_final[col2], 
                           alpha=0.6, s=50, color='blue', label='最后一代', marker='^')
            axes[0].set_xlabel(col_labels.get(col1, col1), fontsize=12)
            axes[0].set_ylabel(col_labels.get(col2, col2), fontsize=12)
            axes[0].set_title(f'{col1} vs {col2}', fontsize=14)
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)
            
            # Makespan vs LoadBalance/LoadBalanceIndex
            col1, col2 = obj_cols[0], obj_cols[2]
            axes[1].scatter(df_first[col1], df_first[col2], 
                           alpha=0.5, s=40, color='red', label='第一代', marker='o')
            axes[1].scatter(df_final[col1], df_final[col2], 
                           alpha=0.6, s=50, color='blue', label='最后一代', marker='^')
            axes[1].set_xlabel(col_labels.get(col1, col1), fontsize=12)
            axes[1].set_ylabel(col_labels.get(col2, col2), fontsize=12)
            axes[1].set_title(f'{col1} vs {col2}', fontsize=14)
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)
            
            # Cost/CostEfficiency vs LoadBalance/LoadBalanceIndex
            col1, col2 = obj_cols[1], obj_cols[2]
            axes[2].scatter(df_first[col1], df_first[col2], 
                           alpha=0.5, s=40, color='red', label='第一代', marker='o')
            axes[2].scatter(df_final[col1], df_final[col2], 
                           alpha=0.6, s=50, color='blue', label='最后一代', marker='^')
            axes[2].set_xlabel(col_labels.get(col1, col1), fontsize=12)
            axes[2].set_ylabel(col_labels.get(col2, col2), fontsize=12)
            axes[2].set_title(f'{col1} vs {col2}', fontsize=14)
            axes[2].legend()
            axes[2].grid(True, alpha=0.3)
        
        plt.suptitle(f"{scheduler_name} - Trial {trial}: 第一代 vs 最后一代 Pareto前沿对比", 
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

## 3. 第一代和最后一代对比可视化（3D）

In [ ]:
# 配置参数（与2D可视化相同）
experiment_dir = "../results/run_21"  # 修改为你的实验目录
scheduler_name = "MOGWO"  # 修改为调度器名称
trial = 1  # 修改为trial编号

# 加载第一代和最后一代的Pareto前沿数据
pareto_dir = os.path.join(experiment_dir, 'pareto_fronts')
file_first = os.path.join(pareto_dir, f"{scheduler_name}_trial_{trial}_first.csv")
file_final = os.path.join(pareto_dir, f"{scheduler_name}_trial_{trial}_final.csv")

# 检查文件是否存在
if not os.path.exists(file_first):
    print(f"⚠️ 第一代数据文件不存在: {file_first}")
    print("请确保已经运行过实验并生成了第一代Pareto前沿数据")
elif not os.path.exists(file_final):
    print(f"⚠️ 最后一代数据文件不存在: {file_final}")
    print("请确保已经运行过实验并生成了最后一代Pareto前沿数据")
else:
    df_first = load_pareto_front(file_first)
    df_final = load_pareto_front(file_final)
    
    if df_first is None or df_first.empty:
        print(f"⚠️ 第一代数据为空: {file_first}")
    elif df_final is None or df_final.empty:
        print(f"⚠️ 最后一代数据为空: {file_final}")
    else:
        # 检测可用的目标列
        obj_cols = get_objective_columns(df_first)
        
        # 获取列名映射（用于显示标签）
        col_labels = {
            'Makespan': 'Makespan',
            'Cost': 'Cost',
            'CostEfficiency': 'CostEfficiency',
            'LoadBalance': 'LoadBalance',
            'LoadBalanceIndex': 'LoadBalanceIndex',
            'ResourceUtilization': 'ResourceUtilization',
            'ResourceWaste': 'ResourceWaste'
        }
        
        if len(obj_cols) == 4:
            # 4个目标：创建4个3D图，每次显示3个目标的不同组合
            fig = plt.figure(figsize=(20, 15))
            
            # 生成所有3个目标的组合（从4个目标中选择3个）
            from itertools import combinations
            triplets = list(combinations(obj_cols, 3))[:4]  # 最多4个组合
            
            for idx, (col1, col2, col3) in enumerate(triplets):
                ax = fig.add_subplot(2, 2, idx + 1, projection='3d')
                ax.scatter(df_first[col1], df_first[col2], df_first[col3], 
                          c='red', alpha=0.4, s=30, label='第一代', marker='o')
                ax.scatter(df_final[col1], df_final[col2], df_final[col3], 
                          c='blue', alpha=0.6, s=50, label='最后一代', marker='^')
                ax.set_xlabel(col_labels.get(col1, col1), fontsize=10)
                ax.set_ylabel(col_labels.get(col2, col2), fontsize=10)
                ax.set_zlabel(col_labels.get(col3, col3), fontsize=10)
                ax.set_title(f'{col1} vs {col2} vs {col3}', fontsize=12)
                ax.legend()
            
            plt.suptitle(f"{scheduler_name} - Trial {trial}: 第一代 vs 最后一代 3D对比（4个目标）", 
                        fontsize=16, fontweight='bold')
        else:
            # 3个目标：单个3D图（向后兼容）
            fig = plt.figure(figsize=(12, 9))
            ax = fig.add_subplot(111, projection='3d')
            
            col1, col2, col3 = obj_cols[0], obj_cols[1], obj_cols[2]
            ax.scatter(df_first[col1], df_first[col2], df_first[col3], 
                      c='red', alpha=0.4, s=30, label='第一代', marker='o')
            ax.scatter(df_final[col1], df_final[col2], df_final[col3], 
                      c='blue', alpha=0.6, s=50, label='最后一代', marker='^')
            
            ax.set_xlabel(col_labels.get(col1, col1), fontsize=12)
            ax.set_ylabel(col_labels.get(col2, col2), fontsize=12)
            ax.set_zlabel(col_labels.get(col3, col3), fontsize=12)
            ax.set_title(f"{scheduler_name} - Trial {trial}: 第一代 vs 最后一代 3D对比", 
                        fontsize=14, fontweight='bold')
            ax.legend()
        
        plt.tight_layout()
        plt.show()

## 4. 学术论文级别的定量对比分析

In [ ]:
# 学术论文级别的对比分析
experiment_dir = "../results/run_21" # 修改为你的实验目录
scheduler_name = "MOPPO"  # 修改为调度器名称
trial = 1  # 修改为trial编号

# 加载数据
pareto_dir = os.path.join(experiment_dir, 'pareto_fronts')
file_first = os.path.join(pareto_dir, f"{scheduler_name}_trial_{trial}_first.csv")
file_final = os.path.join(pareto_dir, f"{scheduler_name}_trial_{trial}_final.csv")

if not os.path.exists(file_first) or not os.path.exists(file_final):
    print(f"⚠️ 数据文件不存在，请检查路径")
else:
    df_first = load_pareto_front(file_first)
    df_final = load_pareto_front(file_final)
    
    if df_first is None or df_first.empty or df_final is None or df_final.empty:
        print("⚠️ 数据为空")
    else:
        print("=" * 80)
        print(f"📊 {scheduler_name} - Trial {trial}: 学术论文级别对比分析")
        print("=" * 80)
        
        # 1. 基本统计信息
        print("\n【1. 基本统计信息】")
        print(f"第一代Pareto解数量: {len(df_first)}")
        print(f"最后一代Pareto解数量: {len(df_final)}")
        print(f"解数量变化: {len(df_final) - len(df_first)} ({((len(df_final) - len(df_first)) / len(df_first) * 100):.2f}%)")
        
        # 2. 各目标函数的统计
        print("\n【2. 各目标函数统计】")
        # 自动检测可用的目标列
        obj_cols = get_objective_columns(df_first)
        objectives = obj_cols  # 使用检测到的目标列
        for obj in objectives:
            first_mean = df_first[obj].mean()
            first_std = df_first[obj].std()
            first_min = df_first[obj].min()
            first_max = df_first[obj].max()
            
            final_mean = df_final[obj].mean()
            final_std = df_final[obj].std()
            final_min = df_final[obj].min()
            final_max = df_final[obj].max()
            
            improvement = ((first_mean - final_mean) / first_mean) * 100
            
            print(f"\n{obj}:")
            print(f"  第一代: 均值={first_mean:.4f}, 标准差={first_std:.4f}, 范围=[{first_min:.4f}, {first_max:.4f}]")
            print(f"  最后一代: 均值={final_mean:.4f}, 标准差={final_std:.4f}, 范围=[{final_min:.4f}, {final_max:.4f}]")
            print(f"  改进: {improvement:.2f}%")
        
        # 3. 评估指标
        print("\n【3. 多目标优化评估指标】")
        
        # 超体积（使用第一代的最差点作为参考点）
        obj_cols_for_ref = get_objective_columns(df_first)
        ref_point = df_first[obj_cols_for_ref].max().values * 1.1
        hv_first = calculate_hypervolume(df_first, ref_point)
        hv_final = calculate_hypervolume(df_final, ref_point)
        hv_improvement = ((hv_final - hv_first) / hv_first) * 100 if hv_first > 0 else 0
        
        print(f"超体积 (Hypervolume):")
        print(f"  第一代: {hv_first:.6f}")
        print(f"  最后一代: {hv_final:.6f}")
        print(f"  改进: {hv_improvement:.2f}%")
        
        # 世代距离（以第一代为参考）
        gd = calculate_generational_distance(df_final, df_first)
        print(f"\n世代距离 (GD, 相对于第一代): {gd:.6f} (越小越好)")
        if len(df_first) == 1:
            print("  ⚠️ 注意：第一代只有1个解，GD值可能不够准确")
        
        # 反向世代距离
        igd = calculate_igd(df_final, df_first)
        print(f"反向世代距离 (IGD): {igd:.6f} (越小越好)")
        if len(df_first) == 1:
            print("  ⚠️ 注意：第一代只有1个解，IGD值可能不够准确")
        
        # 分布性指标
        spread_first = calculate_spread(df_first)
        spread_final = calculate_spread(df_final)
        print(f"\n分布性指标 (Spread):")
        print(f"  第一代: {spread_first:.6f}")
        print(f"  最后一代: {spread_final:.6f}")
        print(f"  变化: {spread_final - spread_first:.6f} (越小越好)")
        if len(df_first) < 2:
            print("  ⚠️ 注意：第一代解数量少于2个，Spread为0（无法计算分布性）")
        if len(df_final) < 2:
            print("  ⚠️ 注意：最后一代解数量少于2个，Spread为0（无法计算分布性）")
        
        # 4. 统计显著性检验
        print("\n【4. 统计显著性检验 (t检验)】")
        if len(df_first) == 1 or len(df_final) == 1:
            print("  ⚠️ 注意：当第一代或最后一代只有1个解时，t检验可能不准确")
        for obj in objectives:
            if len(df_first) == 1 and len(df_final) == 1:
                # 如果都只有1个解，无法进行t检验
                first_val = df_first[obj].iloc[0]
                final_val = df_final[obj].iloc[0]
                improvement = ((first_val - final_val) / first_val) * 100 if first_val != 0 else 0
                print(f"{obj}: 第一代={first_val:.4f}, 最后一代={final_val:.4f}, 改进={improvement:.2f}% (无法进行t检验)")
            else:
                try:
                    t_stat, p_value = stats.ttest_ind(df_first[obj], df_final[obj])
                    significance = "显著" if p_value < 0.05 else "不显著"
                    print(f"{obj}: t={t_stat:.4f}, p={p_value:.6f} ({significance})")
                except Exception as e:
                    print(f"{obj}: 无法进行t检验 ({str(e)})")
        
        print("\n" + "=" * 80)
        print("✅ 分析完成！")
        print("=" * 80)

## 5. 批量生成学术论文对比表格

In [ ]:
# 批量生成学术论文对比表格
experiment_dir = "../results/run_21"  # 修改为你的实验目录
scheduler_name = "MOPPO"  # 修改为调度器名称
num_trials = 10  # trial数量

pareto_dir = os.path.join(experiment_dir, 'pareto_fronts')

results = []

for trial in range(1, num_trials + 1):
    file_first = os.path.join(pareto_dir, f"{scheduler_name}_trial_{trial}_first.csv")
    file_final = os.path.join(pareto_dir, f"{scheduler_name}_trial_{trial}_final.csv")
    
    if not os.path.exists(file_first) or not os.path.exists(file_final):
        continue
    
    df_first = load_pareto_front(file_first)
    df_final = load_pareto_front(file_final)
    
    if df_first is None or df_first.empty or df_final is None or df_final.empty:
        continue
    
    # 检测可用的目标列
    obj_cols = get_objective_columns(df_first)
    
    # 计算指标
    ref_point = df_first[obj_cols].max().values * 1.1
    hv_first = calculate_hypervolume(df_first, ref_point)
    hv_final = calculate_hypervolume(df_final, ref_point)
    gd = calculate_generational_distance(df_final, df_first)
    igd = calculate_igd(df_final, df_first)
    spread_first = calculate_spread(df_first)
    spread_final = calculate_spread(df_final)
    
    # 构建结果字典
    result_dict = {
        'Trial': trial,
        'First_Solutions': len(df_first),
        'Final_Solutions': len(df_final),
        'HV_First': hv_first,
        'HV_Final': hv_final,
        'HV_Improvement': ((hv_final - hv_first) / hv_first) * 100 if hv_first > 0 else 0,
        'GD': gd,
        'IGD': igd,
        'Spread_First': spread_first,
        'Spread_Final': spread_final,
    }
    
    # 各目标函数的改进（动态添加）
    for obj in obj_cols:
        if obj in df_first.columns and obj in df_final.columns:
            first_mean = df_first[obj].mean()
            final_mean = df_final[obj].mean()
            if first_mean > 0:
                imp = ((first_mean - final_mean) / first_mean) * 100
            else:
                imp = 0.0
            result_dict[f'{obj}_Improvement'] = imp
    
    results.append(result_dict)

if results:
    df_results = pd.DataFrame(results)
    
    print("=" * 100)
    print(f"📊 {scheduler_name}: 所有Trial的对比分析结果")
    print("=" * 100)
    print("\n详细数据:")
    print(df_results.to_string(index=False))
    
    print("\n" + "=" * 100)
    print("📈 统计摘要 (均值 ± 标准差):")
    print("=" * 100)
    
    summary = {
        '指标': [],
        '第一代': [],
        '最后一代': [],
        '改进': []
    }
    
    # 超体积
    hv_first_mean = df_results['HV_First'].mean()
    hv_first_std = df_results['HV_First'].std()
    hv_final_mean = df_results['HV_Final'].mean()
    hv_final_std = df_results['HV_Final'].std()
    hv_imp_mean = df_results['HV_Improvement'].mean()
    hv_imp_std = df_results['HV_Improvement'].std()
    
    summary['指标'].append('超体积 (HV)')
    summary['第一代'].append(f"{hv_first_mean:.6f} ± {hv_first_std:.6f}")
    summary['最后一代'].append(f"{hv_final_mean:.6f} ± {hv_final_std:.6f}")
    summary['改进'].append(f"{hv_imp_mean:.2f}% ± {hv_imp_std:.2f}%")
    
    # 世代距离
    gd_mean = df_results['GD'].mean()
    gd_std = df_results['GD'].std()
    summary['指标'].append('世代距离 (GD)')
    summary['第一代'].append('-')
    summary['最后一代'].append(f"{gd_mean:.6f} ± {gd_std:.6f}")
    summary['改进'].append('-')
    
    # 反向世代距离
    igd_mean = df_results['IGD'].mean()
    igd_std = df_results['IGD'].std()
    summary['指标'].append('反向世代距离 (IGD)')
    summary['第一代'].append('-')
    summary['最后一代'].append(f"{igd_mean:.6f} ± {igd_std:.6f}")
    summary['改进'].append('-')
    
    # 分布性
    spread_first_mean = df_results['Spread_First'].mean()
    spread_first_std = df_results['Spread_First'].std()
    spread_final_mean = df_results['Spread_Final'].mean()
    spread_final_std = df_results['Spread_Final'].std()
    summary['指标'].append('分布性 (Spread)')
    summary['第一代'].append(f"{spread_first_mean:.6f} ± {spread_first_std:.6f}")
    summary['最后一代'].append(f"{spread_final_mean:.6f} ± {spread_final_std:.6f}")
    summary['改进'].append(f"{spread_final_mean - spread_first_mean:.6f}")
    
    # 各目标函数改进（动态检测）
    improvement_cols = [col for col in df_results.columns if col.endswith('_Improvement')]
    for col in improvement_cols:
        obj_name = col.replace('_Improvement', '')
        if col in df_results.columns:
            imp_mean = df_results[col].mean()
            imp_std = df_results[col].std()
            summary['指标'].append(f'{obj_name}改进率')
            summary['第一代'].append('-')
            summary['最后一代'].append('-')
            summary['改进'].append(f"{imp_mean:.2f}% ± {imp_std:.2f}%")
    
    df_summary = pd.DataFrame(summary)
    print(df_summary.to_string(index=False))
    
    # 保存为CSV
    output_file = os.path.join(experiment_dir, f"{scheduler_name}_academic_comparison.csv")
    df_results.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n✅ 结果已保存到: {output_file}")
    
    # 保存摘要
    summary_file = os.path.join(experiment_dir, f"{scheduler_name}_academic_summary.csv")
    df_summary.to_csv(summary_file, index=False, encoding='utf-8-sig')
    print(f"✅ 摘要已保存到: {summary_file}")
    
else:
    print("⚠️ 未找到有效数据")

## 6. 学术论文写作建议

### 6.1 对比分析的描述模板

**在论文中描述第一代和最后一代对比时，可以使用以下模板：**

1. **引言部分**：
   > "为了评估算法的收敛性和优化效果，我们对比分析了算法在第一代（初始化后）和最后一代（优化完成后）的Pareto前沿。通过定量指标和可视化分析，验证了算法在多目标优化任务中的有效性。"

2. **结果描述**：
   > "实验结果表明，MO-PPO算法在优化过程中显著改进了Pareto前沿的质量。具体而言，最后一代的Pareto前沿在第一代的基础上，Makespan平均降低了X%，Cost平均降低了Y%，LoadBalance平均降低了Z%。同时，超体积指标从X提升到Y，提升了Z%，表明算法找到了更多高质量的非支配解。"

3. **收敛性分析**：
   > "世代距离(GD)为X，反向世代距离(IGD)为Y，表明算法能够有效收敛到更优的Pareto前沿。分布性指标(Spread)从X降低到Y，说明算法在保持解的质量的同时，也维持了良好的分布性。"

4. **统计显著性**：
   > "t检验结果显示，各目标函数在优化前后的差异均具有统计显著性(p < 0.05)，验证了算法改进的有效性。"

### 6.2 图表建议

1. **2D对比图**：用于展示两个目标之间的权衡关系
2. **3D对比图**：用于展示三个目标的整体分布
3. **统计表格**：包含均值、标准差、改进率等关键指标

### 6.3 关键指标说明

- **超体积(HV)**：越大越好，衡量Pareto前沿的覆盖范围
- **世代距离(GD)**：越小越好，衡量与参考前沿的距离
- **反向世代距离(IGD)**：越小越好，同时考虑收敛性和分布性
- **分布性(Spread)**：越小越好，衡量解的分布均匀程度